# EYES-DEFY-ANEMIA -- Fine-tune Pilot -- Batch 2 (ConvNeXt-Large, MaxViT-T, RegNetY-16GF)

Batch 2 of the partial fine-tuning pilot programme -- extends the first 3 models (ConvNeXt-Base, CoAtNet-3,
EfficientNet-B3, see `classification/.project_memory/13_finetune_pilot_programme.md`) to the next 3
best-performing `new_way` combos by original val F1: `convnext_large_palpebral_new_way` (0.9333),
`maxvit_t_palpebral_new_way` (0.8750), `regnet_y_16gf_palpebral_new_way` (0.8667).

Same "same absolute scale as the first 3 models" design (~8-9M trainable params), applied per-architecture
after inspecting each model's real structure -- full rationale in each engine's own module docstring:
`classification/new_way/Fine_tune/finetune_engine_convnext_large.py`,
`finetune_engine_maxvit_t.py`, `finetune_engine_regnet_y_16gf.py`.

This notebook trains all 3 models in sequence (unlike the first 3, which each got their own notebook), then
builds a cross-model comparison covering all 6 models in the whole programme -- the 3 already-committed
results (read from `Fine_tune/Output/logs/*.json`, arrives via `git clone` below) plus the 3 trained here.

**Data:** TRAIN reads the offline-balanced + online-augmented data (`Offline_data_augmentation/`, arrives via
`git clone`). VAL/TEST read real, unmodified images from the `processed-dataset-clean` Kaggle dataset. The 3
new source checkpoints are NOT in git (gitignored, large binaries) -- they must be attached as a Kaggle
dataset alongside the 3 checkpoints the first batch already uses; see the Data section below.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- run this BEFORE filling in the TODO paths in the Data section below.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# optuna is required even though this notebook never runs a search -- datapreparepipeline/
# trainer_engine.py (reused for compute_metrics/evaluate) imports it unconditionally at module level.
# No timm needed -- both ConvNeXt-Large and MaxViT-T/RegNetY-16GF are torchvision-native.
!pip install -q optuna albumentations

## Data

Two things need to be attached as Kaggle datasets for this notebook to run:

1. **`processed-dataset-clean`** (same dataset every `new_way/` notebook uses) -- provides the real
   VAL/TEST images plus `splits.csv`/`extraction_log.csv`.
2. **The `checkpoints` dataset**, extended with 3 more files. This is the same dataset the first batch's
   3 notebooks already use -- `best_convnext_large_palpebral_new_way.pth`, `best_maxvit_t_palpebral_new_way.pth`,
   and `best_regnet_y_16gf_palpebral_new_way.pth` need to be added to it as a new version (re-zip locally with
   all 6 `.pth` files and re-upload to the same Kaggle dataset -- the mount path/slug doesn't change).

Check the `/kaggle/input` listing above and fill in both TODO paths below before running.

In [ ]:
import shutil
from pathlib import Path

# TODO: verify against the /kaggle/input listing cell above before running.
PROCESSED_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in PROCESSED_SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
# TODO: verify against the /kaggle/input listing cell above before running -- this is the
# checkpoint dataset (see markdown above), not the same as PROCESSED_SRC_DIR. Only the 3 NEW
# checkpoints below are required by this notebook; the other 3 (if present in the same dataset)
# are simply ignored here.
CHECKPOINT_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/checkpoints")
CHECKPOINT_DST_DIR = Path("classification/new_way/Output/version1/checkpoints")
CHECKPOINT_DST_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for item in CHECKPOINT_SRC_DIR.rglob("*.pth"):
    shutil.copy2(item, CHECKPOINT_DST_DIR / item.name)
    copied += 1
    print(f"  copied {item.name} ({item.stat().st_size / 1e6:.1f} MB)")
print(f"\n{copied} checkpoint file(s) staged to {CHECKPOINT_DST_DIR}")

In [ ]:
# Fails loudly here, not deep inside training, if either data source above is missing/misconfigured.
manifest_path = Path("classification/new_way/Offline_data_augmentation/manifest.csv")
assert manifest_path.exists(), (
    f"{manifest_path} not found -- Offline_data_augmentation/ should have arrived via git clone. "
    "Was it actually committed and pushed?"
)

required_checkpoints = [
    "best_convnext_large_palpebral_new_way.pth",
    "best_maxvit_t_palpebral_new_way.pth",
    "best_regnet_y_16gf_palpebral_new_way.pth",
]
for name in required_checkpoints:
    p = CHECKPOINT_DST_DIR / name
    assert p.exists(), (
        f"{p} not found -- fix CHECKPOINT_SRC_DIR above to point at the real "
        "checkpoint-dataset mount path (check the /kaggle/input listing cell), and confirm "
        "the dataset actually contains this batch's 3 new checkpoints."
    )
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")
print(f"\nAll data sources present: {manifest_path}, and all {len(required_checkpoints)} required checkpoints.")

## Sanity checks

Builds each fine-tune model for real (loads its checkpoint, applies the freeze/unfreeze split) and confirms
the trainable-parameter count matches what was verified locally, before any real training starts.

In [ ]:
import sys
sys.path.insert(0, "classification/new_way/Fine_tune")

import finetune_engine_convnext_large as fe_cl

model = fe_cl.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"ConvNeXt-Large trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "18,964,993", (
    f"Expected 18,964,993 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_maxvit_t as fe_mt

model = fe_mt.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"MaxViT-T trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "8,962,465", (
    f"Expected 8,962,465 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_regnet_y_16gf as fe_rn

model = fe_rn.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"RegNetY-16GF trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "9,153,649", (
    f"Expected 9,153,649 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

## Output syncing

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate Fine_tune/Output/{checkpoints,logs,plots}/ into a single top-level
    /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/batch2_finetune_results.zip."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/new_way/Fine_tune/Output") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/batch2_finetune_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- ConvNeXt-Large

Reproduces the locally-designed configuration -- fixed hyperparameters (no Optuna), discriminative LRs,
val-F1-tracked scheduler/early-stopping. Prints the baseline (pre-fine-tune) metrics first, then trains, then
prints a before/after comparison on the sealed test set.

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_convnext_large_palpebral.py
sync_outputs()

## Training -- MaxViT-T

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_maxvit_t_palpebral.py
sync_outputs()

## Training -- RegNetY-16GF

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_regnet_y_16gf_palpebral.py
sync_outputs()

## Cross-model comparison (all 6 models)

Combines this batch's 3 fresh results with the first batch's 3 already-committed results (their
`*_history.json` files arrived via this notebook's own `git clone`, since they're already on GitHub) into
one set of comparison plots + a summary table -- the full picture of the whole fine-tuning programme, not
two disconnected halves.

In [ ]:
import finetune_common as fc

FINE_TUNE_LOGS_DIR = Path("classification/new_way/Fine_tune/Output/logs")
FINE_TUNE_PLOTS_DIR = Path("classification/new_way/Fine_tune/Output/plots")

ALL_SIX_MODEL_NAMES = [
    "convnext_base_palpebral_new_way_finetune_block3_v2",
    "coatnet_3_palpebral_new_way_finetune_attn",
    "efficientnet_b3_forniceal_palpebral_new_way_finetune_block_v2",
    "convnext_large_palpebral_new_way_finetune_block3",
    "maxvit_t_palpebral_new_way_finetune_lastlayer",
    "regnet_y_16gf_palpebral_new_way_finetune_finalconv",
]

histories = {name: fc.load_finetune_history(FINE_TUNE_LOGS_DIR, name) for name in ALL_SIX_MODEL_NAMES}
comparison_paths = fc.plot_cross_model_comparison(histories, FINE_TUNE_PLOTS_DIR, FINE_TUNE_LOGS_DIR)
print(comparison_paths)

sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` and zipped to `/kaggle/working/batch2_finetune_results.zip`.
Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All.

The headline number for each new model is in its own `{model_name}_history.json`'s `best_val_f1` and
`test_metrics` -- compare against this notebook's own printed before/after blocks, and against the 6-model
cross-model comparison plots/table (`cross_model_val_f1_comparison.png`, `cross_model_test_metrics_comparison.png`,
`cross_model_auc_gap_comparison.png`, `cross_model_comparison_table.md`).

In [ ]:
print("Final contents of /kaggle/working/outputs:")
for f in sorted(Path("/kaggle/working/outputs").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/kaggle/working/outputs')}  ({f.stat().st_size / 1e6:.2f} MB)")

zip_path = Path("/kaggle/working/batch2_finetune_results.zip")
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")